In [3]:
# math_datasets_eda_clean.py
"""A cleaned‑up, modular version of the exploratory analysis notebook for
OpenWebMath, ASDiv, ParaMAWPS, DMath, and MathInstruct.

Each dataset is analysed in an isolated function that:
  • loads the data (or a sample, where appropriate),
  • computes token statistics,
  • saves informative plots to both a local and a cloud directory, and
  • returns problem/solution token lists for downstream comparison.

The main script runs all analyses, assembles a cross‑dataset summary table,
plots overall comparisons, and performs a complexity‑ratio study.

All variables are scoped locally to avoid name collisions and duplicate
side‑effects, eliminating the IndentationError caused by accidental code
repetition in the original notebook.
"""
from __future__ import annotations

import json
import os
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import load_dataset
from nltk.tokenize import word_tokenize

###############################################################################
# Configuration
###############################################################################

# ---- File system paths -----------------------------------------------------
ASDIV_PATH = Path(
    "/Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/data/curriculum_learning/1_ASDiv/ASDiv.xml"
)
PARAMAWPS_PATH = Path(
    "/Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/data/curriculum_learning/2_ParaMAWPS/ParaMAWPS_trainset.json"
)
DMATH_PATH = Path(
    "/Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/data/curriculum_learning/4_Dmath/dmath_train.json"
)

LOCAL_PLOT_DIR = Path("domain_plots")
CLOUD_PLOT_DIR = Path(
    "/Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/plots/plots_images"
)

# ---- Runtime parameters ----------------------------------------------------
SAMPLE_SIZE_OPENWEBMATH = 10_000  # adjust as needed
SAMPLE_SIZE_MATHINSTRUCT = 5_000  # adjust as needed

###############################################################################
# Initialisation
###############################################################################

nltk.download("punkt", quiet=True)
plt.style.use("ggplot")
sns.set(style="whitegrid")

for d in (LOCAL_PLOT_DIR, CLOUD_PLOT_DIR):
    d.mkdir(parents=True, exist_ok=True)

###############################################################################
# Helper utilities
###############################################################################

def count_tokens(text: str | None) -> int:
    """Count whitespace‑separated tokens in *text* using NLTK."""
    return len(word_tokenize(text)) if isinstance(text, str) else 0


def _savefig(fig: plt.Figure, name: str) -> None:
    """Save *fig* to both local and cloud plot directories."""
    local_path = LOCAL_PLOT_DIR / name
    cloud_path = CLOUD_PLOT_DIR / name
    fig.savefig(local_path, bbox_inches="tight")
    fig.savefig(cloud_path, bbox_inches="tight")

###############################################################################
# Per‑dataset analysis functions
###############################################################################

def analyze_openwebmath(sample_size: int) -> Tuple[List[int], List[int]]:
    print("\nLoading OpenWebMath …")
    ds = load_dataset("open-web-math/open-web-math", split=f"train[:{sample_size}]")
    print(f"Loaded {len(ds):,} examples")

    q_tokens, a_tokens, domains, sources = [], [], [], []

    for row in ds:
        q_tokens.append(count_tokens(row.get("question", "")))
        a_tokens.append(count_tokens(row.get("answer", "")))

        url = row.get("url", "")
        if url:
            parts = url.split("/")
            if len(parts) > 2:
                domains.append(parts[2])
        if src := row.get("source", ""):
            sources.append(src)

    # --- Plots ----------------------------------------------------------------
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    sns.histplot(q_tokens, kde=True, ax=ax[0])
    ax[0].set_title("Question token distribution")
    ax[0].set_xlabel("Number of tokens")
    sns.histplot(a_tokens, kde=True, ax=ax[1])
    ax[1].set_title("Answer token distribution")
    ax[1].set_xlabel("Number of tokens")
    fig.suptitle("OpenWebMath token counts")
    _savefig(fig, "openwebmath_tokens.png")
    plt.close(fig)

    if domains:
        _barplot_counter(Counter(domains), "openwebmath_domains.png", "Top domains – OpenWebMath")
    if sources:
        _barplot_counter(Counter(sources), "openwebmath_sources.png", "Top sources – OpenWebMath")

    return q_tokens, a_tokens


def analyze_asdiv() -> Tuple[List[int], List[int]]:
    print("\nLoading ASDiv …")
    tree = ET.parse(ASDIV_PATH)
    root = tree.getroot()

    body_toks, q_toks, a_toks, sol_types, grades = [], [], [], [], []

    for prob in root.findall(".//Problem"):
        body = prob.findtext("Body", default="")
        question = prob.findtext("Question", default="")
        answer = prob.findtext("Answer", default="")
        sol_type = prob.findtext("Solution-Type", default="")
        grade = prob.get("Grade", "")

        body_toks.append(count_tokens(body))
        q_toks.append(count_tokens(question))
        a_toks.append(count_tokens(answer))
        sol_types.append(sol_type)
        grades.append(grade)

    # --- Plots ----------------------------------------------------------------
    # Token distribution
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.histplot([b + q for b, q in zip(body_toks, q_toks)], bins=30, label="Problem", color="steelblue")
    sns.histplot(a_toks, bins=30, label="Answer", color="salmon")
    ax.set_title("ASDiv token distribution")
    ax.set_xlabel("Number of tokens")
    ax.legend()
    _savefig(fig, "asdiv_tokens.png")
    plt.close(fig)

    _barplot_counter(Counter(sol_types), "asdiv_solution_types.png", "Solution types – ASDiv")
    _barplot_counter(Counter(grades), "asdiv_grades.png", "Grade distribution – ASDiv", rotate=False)

    return [b + q for b, q in zip(body_toks, q_toks)], a_toks


def analyze_paramawps() -> Tuple[List[int], List[int]]:
    print("\nLoading ParaMAWPS …")
    data = json.loads(PARAMAWPS_PATH.read_text())

    text_toks, eq_toks, ops = [], [], []

    for item in data:
        text = item.get("original_text") or item.get("segmented_text", "")
        eq = item.get("equation", "")
        text_toks.append(count_tokens(text))
        eq_toks.append(count_tokens(eq))

        # crude op detection
        if "+" in eq:
            ops.append("addition")
        elif "-" in eq and "*" not in eq:
            ops.append("subtraction")
        elif "*" in eq and "/" not in eq:
            ops.append("multiplication")
        elif "/" in eq:
            ops.append("division")
        else:
            ops.append("other")

    # --- Plots ----------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.histplot(text_toks, bins=30, label="Problem text", color="steelblue")
    sns.histplot(eq_toks, bins=30, label="Equation", color="salmon")
    ax.set_title("ParaMAWPS token distribution")
    ax.set_xlabel("Number of tokens")
    ax.legend()
    _savefig(fig, "paramawps_tokens.png")
    plt.close(fig)

    _barplot_counter(Counter(ops), "paramawps_operations.png", "Equation operation types – ParaMAWPS")

    return text_toks, eq_toks


def analyze_dmath() -> Tuple[List[int], List[int]]:
    print("\nLoading DMath …")
    data: Dict[str, Dict] = json.loads(DMATH_PATH.read_text())

    q_toks, a_toks, cats = [], [], []

    for item in data.values():
        q_toks.append(count_tokens(item.get("question_en", "")))
        a_toks.append(count_tokens(item.get("answer_en", "")))
        cats.append(item.get("category", "Unknown"))

    # --- Plots ----------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.histplot(q_toks, bins=30, label="Question", color="steelblue")
    sns.histplot(a_toks, bins=30, label="Answer", color="salmon")
    ax.set_title("DMath token distribution")
    ax.set_xlabel("Number of tokens")
    ax.legend()
    _savefig(fig, "dmath_tokens.png")
    plt.close(fig)

    _barplot_counter(Counter(cats), "dmath_categories.png", "Top categories – DMath")

    return q_toks, a_toks


def analyze_mathinstruct(sample_size: int) -> Tuple[List[int], List[int]]:
    print("\nLoading MathInstruct …")
    ds = load_dataset("TIGER-Lab/MathInstruct", split="train")
    ds = ds.select(range(min(sample_size, len(ds))))
    print(f"Sampled {len(ds):,} examples")

    instr_toks, in_toks, out_toks, sources = [], [], [], []

    for row in ds:
        instr_toks.append(count_tokens(row.get("instruction", "")))
        in_toks.append(count_tokens(row.get("input", "")))
        out_toks.append(count_tokens(row.get("output", "")))
        if src := row.get("source", ""):
            sources.append(src)

    # --- Plots ----------------------------------------------------------------
    prob_toks = [i + j for i, j in zip(instr_toks, in_toks)]

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.histplot(prob_toks, bins=30, label="Problem", color="steelblue")
    sns.histplot(out_toks, bins=30, label="Solution", color="salmon")
    ax.set_title("MathInstruct token distribution")
    ax.set_xlabel("Number of tokens")
    ax.legend()
    _savefig(fig, "mathinstruct_tokens.png")
    plt.close(fig)

    if sources:
        _barplot_counter(Counter(sources), "mathinstruct_sources.png", "Top sources – MathInstruct")

    return prob_toks, out_toks

###############################################################################
# Generic plotting helpers
###############################################################################

def _barplot_counter(counter: Counter, filename: str, title: str, *, rotate: bool = True) -> None:
    labels, counts = zip(*counter.most_common(10))
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x=list(counts), y=list(labels), ax=ax)
    if rotate:
        ax.set_yticklabels(labels, rotation=0)
    ax.set_title(title)
    ax.set_xlabel("Count")
    _savefig(fig, filename)
    plt.close(fig)

###############################################################################
# Summary & complexity analysis
###############################################################################

def summarise(datasets: Dict[str, Tuple[List[int], List[int]]]) -> None:
    summary_rows = []
    for name, (p_toks, s_toks) in datasets.items():
        summary_rows.append(
            {
                "Dataset": name,
                "Problem Tokens (Avg)": f"{np.mean(p_toks):.2f}",
                "Solution Tokens (Avg)": f"{np.mean(s_toks):.2f}",
                "Samples": len(p_toks),
            }
        )

    df = pd.DataFrame(summary_rows)
    print("\n=== Token‑count summary ===")
    print(df.to_string(index=False))

    # Bar‑plot comparison
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(df))
    width = 0.35
    ax.bar(x - width / 2, df["Problem Tokens (Avg)"].astype(float), width, label="Problem")
    ax.bar(x + width / 2, df["Solution Tokens (Avg)"].astype(float), width, label="Solution")
    ax.set_xticks(x)
    ax.set_xticklabels(df["Dataset"])
    ax.set_ylabel("Average token count")
    ax.set_xlabel("Dataset")
    ax.set_title("Average token count per dataset")
    ax.legend()
    _savefig(fig, "token_comparison.png")
    plt.close(fig)

    # Complexity ratio
    ratio_rows = []
    for name, (p_toks, s_toks) in datasets.items():
        ratios = [s / p for p, s in zip(p_toks, s_toks) if p > 0]
        ratio_rows.append(
            {
                "Dataset": name,
                "Avg Solution/Problem Ratio": f"{np.mean(ratios):.2f}",
                "Median Ratio": f"{np.median(ratios):.2f}",
            }
        )

    ratio_df = pd.DataFrame(ratio_rows)
    print("\n=== Solution/Problem complexity ratio ===")
    print(ratio_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(ratio_df["Dataset"], ratio_df["Avg Solution/Problem Ratio"].astype(float))
    ax.axhline(1.0, color="red", linestyle="--")
    ax.text(-0.4, 1.05, "1:1 ratio", color="red")
    ax.set_ylabel("Average solution/problem token ratio")
    ax.set_xlabel("Dataset")
    ax.set_title("Solution complexity across datasets")
    _savefig(fig, "solution_problem_ratio.png")
    plt.close(fig)

###############################################################################
# Main entry point
###############################################################################

def main() -> None:
    datasets: Dict[str, Tuple[List[int], List[int]]] = {}

    try:
        datasets["OpenWebMath"] = analyze_openwebmath(SAMPLE_SIZE_OPENWEBMATH)
    except Exception as e:
        print(f"WARNING: OpenWebMath failed: {e}")

    try:
        datasets["ASDiv"] = analyze_asdiv()
    except Exception as e:
        print(f"WARNING: ASDiv failed: {e}")

    try:
        datasets["ParaMAWPS"] = analyze_paramawps()
    except Exception as e:
        print(f"WARNING: ParaMAWPS failed: {e}")

    try:
        datasets["DMath"] = analyze_dmath()
    except Exception as e:
        print(f"WARNING: DMath failed: {e}")

    try:
        datasets["MathInstruct"] = analyze_mathinstruct(SAMPLE_SIZE_MATHINSTRUCT)
    except Exception as e:
        print(f"WARNING: MathInstruct failed: {e}")

    if datasets:
        summarise(datasets)
        print(
            f"\nEDA complete. Plots are saved in:\n  – {LOCAL_PLOT_DIR.resolve()}\n  – {CLOUD_PLOT_DIR.resolve()}"
        )
    else:
        print("No dataset analysis succeeded – nothing to summarise.")


if __name__ == "__main__":
    main()

Using the latest cached version of the dataset since open-web-math/open-web-math couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/jonathan/.cache/huggingface/datasets/open-web-math___open-web-math/default/0.0.0/fde8ef8de2300f5e778f56261843dab89f230815 (last modified on Mon Mar 10 17:12:17 2025).



Loading OpenWebMath …
Loaded 10,000 examples


/var/folders/cw/nsgz30_17pq84g4s1jh0lgtw0000gp/T/ipykernel_79112/4065147600.py:272: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(labels, rotation=0)



Loading ASDiv …


/var/folders/cw/nsgz30_17pq84g4s1jh0lgtw0000gp/T/ipykernel_79112/4065147600.py:272: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(labels, rotation=0)



Loading ParaMAWPS …


/var/folders/cw/nsgz30_17pq84g4s1jh0lgtw0000gp/T/ipykernel_79112/4065147600.py:272: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(labels, rotation=0)



Loading DMath …


/var/folders/cw/nsgz30_17pq84g4s1jh0lgtw0000gp/T/ipykernel_79112/4065147600.py:272: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(labels, rotation=0)



Loading MathInstruct …
